In [ ]:
import shutil
import subprocess
from pathlib import Path

import pandas as pd

OUT = Path("results/rates")
OUT.mkdir(parents=True, exist_ok=True)
rates_path = OUT / "site_rates.csv"
if not rates_path.exists():
    hyphy = shutil.which("hyphy")
    if hyphy is None:
        raise SystemExit("HyPhy is not installed and results/rates/site_rates.csv is absent")
    subprocess.run([
        hyphy, "CPU=4", "LEISR",
        "--alignment", "results/alignment/colap_trimmed_aligned.fasta",
        "--tree", "results/phylogeny/colap_ml.treefile",
        "--type", "Protein", "--model", "WAG", "--rate-variation", "No",
    ], check=True)
rates = pd.read_csv(rates_path)
structure = pd.read_csv("results/structure/structural_sites.csv")
occupied = rates[rates["is_insertion"].astype(str).str.lower().isin(["false", "0"])].copy()
occupied["uniprot_position"] = pd.to_numeric(occupied["uniprot_position"], errors="coerce")
occupied = occupied.dropna(subset=["uniprot_position"])
occupied["uniprot_position"] = occupied["uniprot_position"].astype(int)
joined = structure.merge(
    occupied[["uniprot_position", "relative_rate", "shannon_entropy_bits", "fully_conserved", "n_distinct_aa"]],
    on="uniprot_position",
    how="left",
)
if joined["relative_rate"].isna().any():
    raise SystemExit("a structural position has no LEISR rate")
joined.to_csv(OUT / "sites_with_structure.csv", index=False)
